# Future instruction
When it comes to updating database objects at the start of the season, I need to ensure each df fits with its corresponding db table.
In terms of "fit":
    <br>&emsp;If replace, ensure the new object contains all data and none is lost
    <br>&emsp;If append, ensure the df to be unioned is correct

In [1]:
import pandas as pd; pd.set_option('display.max_columns', 500)
from sqlalchemy.dialects.postgresql.base import PGDialect; PGDialect._get_server_version_info = lambda *args: (9, 2)
from dataHub import dataHub
dh = dataHub()

db_con = dh.db_connect('cockroach')
postgre = dh.db_connect('postgre')
fty_con = dh.fty_api_con()

# Game Schedule
Use update_past_game_schedule() to update records from last season. Use get_next_game_schedule() to obtain schedule for upcoming season. Run both functions at the start of each season

In [2]:
from time import sleep
from pandas import DataFrame, concat, read_sql_query, to_datetime, to_numeric
from numpy import where
import snakecase
from requests import get
from pandasql import sqldf; pysqldf = lambda q: sqldf(q, globals())

from nba_api.stats.endpoints import leaguegamelog
from nba_api.stats.static import players, teams
from nba_api.stats.library.parameters import Season

In [ ]:
col_order = read_sql_query("SELECT column_name FROM util.table_column_order WHERE table_name = 'league_game_schedule' ORDER BY column_order", db_con)['column_name'].to_list()

print('\n--------------------- historical_league_game_schedule')
df = DataFrame() 
for type_season in ['Regular Season', 'Pre Season', 'Playoffs', 'All Star', 'All-Star']:
    hist_game_schedule = leaguegamelog.LeagueGameLog(season_type_all_star=type_season, season=Season.current_season_year-1)
    hist_game_schedule = hist_game_schedule.get_data_frames()[0]
    hist_game_schedule['type_season'] = type_season
    df = concat([df, hist_game_schedule], ignore_index=True)
    sleep(1)

df = df.groupby(['GAME_ID']).head(1)
df['GAME_ID'] = to_numeric(df['GAME_ID'])
df['slug_matchup'] = df['MATCHUP']
df['opponent'] = df['MATCHUP'].str.replace(r'[ @ | vs. ]', '', regex=True)
df['opponent'] = df.apply(lambda x: x['opponent'].replace(x['TEAM_ABBREVIATION'], ''), axis=1)
df['slug_team_winner'] = where(df['WL'] == 'W', df['TEAM_ABBREVIATION'], df['opponent'])
df['slug_team_loser'] = where(df['WL'] == 'L', df['TEAM_ABBREVIATION'], df['opponent'])
df['slug_season'] = Season.previous_season
df = df.rename(snakecase.convert, axis='columns')
df = df[col_order]

df_t = read_sql_query("SELECT * FROM nba.league_game_schedule WHERE slug_season != '{}'".format(Season.previous_season), postgre)
df = concat([df_t, df], ignore_index=True)
df.to_sql('league_game_schedule', db_con, schema='nba', index=False, if_exists='replace')